In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import json
from collections import Counter, defaultdict
from pathlib import Path
from zipfile import ZipFile, ZIP_STORED

import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

In [3]:
DRIVE = Path('/content/drive/MyDrive/WasteScope')
ARCHIVE_DIR = DRIVE / 'raw_archives'
TASKS = {
    'detection': ARCHIVE_DIR / 'detection_task.zip',
    'tracking': ARCHIVE_DIR / 'tracking_task.zip',
    'segmentation': ARCHIVE_DIR / 'segmentation_task.zip'
}
CLASSES = [
    'container_yellow', 'container_green', 'container_default',
    'container_blue', 'container_oil', 'container_battery',
    'container_biodegradable'
]

In [5]:
with ZipFile(TASKS["segmentation"]) as archive:
    json_name = next(name for name in archive.namelist() if Path(name).name == 'labels.json')
    coco = json.loads(archive.read(json_name).decode("utf-8-sig"))

categories = {item["id"]: item["name"] for item in coco["categories"]}
overflow_id = next(cid for cid, name in categories.items() if name.lower() == 'overflow')
first_container = min(cid for cid, name in categories.items() if name.startswith('container_'))

In [6]:
DETECTION_MAP = {
    cid - 1: CLASSES.index(name) for cid, name in categories.items() if name in CLASSES
}
TRACKING_MAP = {
    cid - first_container: CLASSES.index(name)
    for cid, name in categories.items() if name in CLASSES
}

DETECTION_SKIP = {cid - 1 for cid, name in categories.items() if name not in CLASSES}
TRACKING_SKIP = {
    cid - first_container for cid, name in categories.items()
    if name.startswith('container_') and name not in CLASSES
}

In [ ]:
images, labels = {}, {}

for task, path in TASKS.items():
    with ZipFile(path) as archive:
        image_names = sorted(name for name in archive.namelist() if name.lower().endswith(".png"))
        label_names = sorted(name for name in archive.namelist() if name.lower().endswith(".txt") and Path(name).parent.name == "labels")
    
    images[task] = {Path(name).stem: name for name in image_names}
    labels[task] = {Path(name).stem: name for name in label_names}


display(pd.DataFrame([
    {
        'Task': task,
        'Images': len(files),
        'Videos': len({stem.split('_')[0] for stem in files})
    }
    for task, files in images.items()
]).set_index('Task'))

display(pd.DataFrame([
    {
        'Source category': name, 'JSON ID': cid,
        'Detection TXT ID': cid - 1,
        'Tracking TXT ID': cid - first_container if name.startswith('container_') else 'N/A',
        'Target ID': CLASSES.index(name) if name in CLASSES else 'Excluded',
    }
    for cid, name in categories.items()
]))

,Images,Videos
Task,,
detection,36478,172
tracking,10022,93
segmentation,7230,38


,Source category,JSON ID,Detection TXT ID,Tracking TXT ID,Target ID
0,Overflow,1,0,N/A,Excluded
1,container_yellow,2,1,0,0
2,container_green,3,2,1,1
3,container_default,4,3,2,2
4,container_blue,5,4,3,3
5,container_ash,6,5,4,Excluded
6,container_oil,7,6,5,4
7,container_battery,8,7,6,5
8,container_biodegradable,9,8,7,6


In [9]:
def container_summary(task, mapping, excluded, field_count):
    instances = Counter()
    class_images = defaultdict(set)
    class_videos = defaultdict(set)
    
    with ZipFile(TASKS[task]) as archive:
        for stem, member in tqdm(sorted(labels[task].items())):
            for line in archive.read(member).decode("utf-8-sig").splitlines():
                fields = line.split()
                if not fields:
                    continue
                
                if len(fields) != field_count:
                    raise ValueError(f'Invalid label format: {member}')
                
                raw_id = int(fields[0])
                if raw_id in excluded:
                    continue
                
                cid = mapping[raw_id]
                instances[cid] += 1
                class_images[cid].add(stem)
                class_videos[cid].add(stem.split('_')[0])
        
    return pd.DataFrame([
        {
            'ID': cid, 'Class': name,
            'Instances': instances[cid],
            'Images': len(class_images[cid]),
            'Videos': len(class_videos[cid])
        }
        for cid, name in enumerate(CLASSES)
    ]).set_index('ID')

In [10]:
print("Detection: seven target classes")
display(container_summary('detection', DETECTION_MAP, DETECTION_SKIP, 5))

print("Tracking: seven target classes")
display(container_summary('tracking', TRACKING_MAP, TRACKING_SKIP, 7))

Detection: seven target classes


  0%|          | 0/36478 [00:00<?, ?it/s]

,Class,Instances,Images,Videos
ID,,,,
0,container_yellow,10208,9768,75
1,container_green,9918,9918,78
2,container_default,28346,14346,114
3,container_blue,10930,10278,77
4,container_oil,1873,1873,15
5,container_battery,2595,2535,28
6,container_biodegradable,1734,1734,10


Tracking: seven target classes


  0%|          | 0/10022 [00:00<?, ?it/s]

,Class,Instances,Images,Videos
ID,,,,
0,container_yellow,5010,4828,47
1,container_green,5141,5141,51
2,container_default,15862,8647,86
3,container_blue,5479,5131,49
4,container_oil,890,890,8
5,container_battery,1294,1288,17
6,container_biodegradable,523,523,5
